# 9 . Squeeze Momentum + WaveTrend [LazyBear]

Faithful ports of two TradingView LazyBear studies, shown in a **three-pane chart** like TradingView:

- **Top:** price candles (+ EMAs).
- **Middle — Squeeze Momentum (SQZMOM_LB):** momentum histogram coloured by sign & rising/falling
  (🟢 up-rising · 🟡 up-fading · 🔴 down-rising · 🟣 down-fading) with **squeeze dots** on the zero
  line (**black = ON**, gray = off, blue = none).
- **Bottom — WaveTrend (WT_CROSS_LB):** WT1 (green) / WT2 (red) waves, the blue WT1−WT2 area, the
  Over-Bought/Sold guide lines (±60 / ±53), and **cross dots** where WT1 crosses WT2
  (green = bullish cross, red = bearish).

Extras: a **cursor crosshair** (horizontal + vertical lines) to read price / indicator values off
the axes, and a **clickable legend** to toggle any element on/off.

> Both are exact ports of the Pine studies (matching each dialog's inputs). Squeeze keeps LazyBear's
> quirk where BB MultFactor is present-but-unused (pass `faithful=False` to override).

## 0 . Setup

In [ ]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import pandas as pd
from quant.data import get_ohlcv
from quant.indicators import add_squeeze_momentum, add_wavetrend
from quant.viz import studies_figure, open_in_browser, CROSSHAIR_JS
pd.set_option('display.max_rows', 200)
print('quant', quant.__version__)

## 1 . Data
BTC/USD to match the screenshot; switch to `('XAUUSD','dukascopy','cfd')` or `('PAXGUSDT',...)` for
gold. Keep the window short so 1m candles are legible (like a TradingView screen).

In [ ]:
SYMBOL, SOURCE, MARKET, TF = 'XAUUSD','dukascopy','cfd', '1m'
START, END = '2026-07-24', '2026-07-25'
df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} {TF} bars  {df.open_time.min()} -> {df.open_time.max()}')

## 2 . Indicator settings (from the TradingView dialog)

In [ ]:
# --- Squeeze Momentum (SQZMOM_LB) inputs ---
BB_LENGTH, BB_MULT = 20, 2.0     # BB Length / BB MultFactor
KC_LENGTH, KC_MULT = 20, 1.5     # KC Length / KC MultFactor
USE_TRUERANGE = True             # Use TrueRange (KC)

# --- WaveTrend (WT_CROSS_LB) inputs ---
WT_CHANNEL, WT_AVERAGE = 10, 21  # Channel Length / Average Length
WT_OB1, WT_OB2 = 60, 53          # Over Bought Level 1 / 2
WT_OS1, WT_OS2 = -60, -53        # Over Sold Level 1 / 2

d = add_squeeze_momentum(df, BB_LENGTH, BB_MULT, KC_LENGTH, KC_MULT, USE_TRUERANGE)
d = add_wavetrend(d, WT_CHANNEL, WT_AVERAGE, WT_OB1, WT_OB2, WT_OS1, WT_OS2)
print('squeeze cols:', [c for c in d.columns if c.startswith('sqz')])
print('wavetrend cols:', [c for c in d.columns if c.startswith('wt')])
print('squeeze ON bars:', int(d.sqz_on.sum()), '| WT crosses up/down:', int(d.wt_cross_up.sum()), int(d.wt_cross_down.sum()))
d[['t','close','sqz_mom','sqz_on','wt1','wt2']].tail(6)

## 3 . Plot — price + Squeeze Momentum + WaveTrend (opens in a browser tab)
Three panes like TradingView. Move the mouse for the **crosshair**; click **legend** items to
toggle elements; hover the squeeze/WT dots for details.

In [ ]:
fig, d = studies_figure(
    df, emas=[20, 50], squeeze=True, wavetrend=True,
    title=f'{SYMBOL} {TF} - Squeeze Momentum + WaveTrend [LazyBear]',
    bb_length=BB_LENGTH, bb_mult=BB_MULT, kc_length=KC_LENGTH, kc_mult=KC_MULT, use_tr=USE_TRUERANGE,
    wt_channel=WT_CHANNEL, wt_average=WT_AVERAGE,
    wt_ob1=WT_OB1, wt_ob2=WT_OB2, wt_os1=WT_OS1, wt_os2=WT_OS2)
# CROSSHAIR_JS adds the full-height vertical cursor line spanning ALL panes
print('chart ->', open_in_browser(fig, 'studies_chart.html', extra_js=CROSSHAIR_JS))

## 4 . Reading them
**Squeeze Momentum** — *black dots* = Bollinger Bands inside the Keltner Channel (volatility
compressed, energy building); when the dot turns *gray* the squeeze fires — the histogram
colour/direction hints the way. Green/yellow = bullish momentum (rising/fading); red/purple = bearish.

**WaveTrend** — an overbought/oversold momentum oscillator. Look for **WT1 crossing WT2** (the dots)
in the **oversold zone (≤ −53/−60)** for potential longs, or in the **overbought zone (≥ 53/60)** for
potential shorts. The blue WT1−WT2 area shows momentum strength/direction. A common combo: take a
WaveTrend cross **in the direction of a Squeeze release**.

> All inputs live in section 2 — change them and re-run; the chart follows. Set `squeeze=False` or
> `wavetrend=False` in `studies_figure(...)` to show just one pane.